In [ ]:
from pymongo import MongoClient
from pymongo.errors import OperationFailure, PyMongoError
import os
from dotenv import load_dotenv
import certifi
from datetime import datetime
from bson.objectid import ObjectId

load_dotenv()

class AtlasMigrator:
    def __init__(self):
        self.client = None
        self.db = None
        
        # Configuration
        self.ATLAS_URI = os.getenv("DATABASE_URL")
        self.DB_NAME = os.getenv("DB_NAME", "odim")
        
        # SSL Configuration
        self.ssl_kwargs = {
            'tls': True,
            'tlsCAFile': certifi.where(),
            'connectTimeoutMS': 30000,
            'serverSelectionTimeoutMS': 10000
        }

    def connect(self):
        """Establish secure Atlas connection"""
        print(self.ATLAS_URI)
        print(self.DB_NAME)
        try:
            self.client = MongoClient(self.ATLAS_URI, **self.ssl_kwargs)
            self.db = self.client[self.DB_NAME]
            self.db.command('ping')  # Test connection
            print("Verified Atlas connection")
            return True
        except PyMongoError as e:
            print(f"Connection failed: {str(e)}")
            return False

    def check_review_capture_with_android_app(self):
        captures = self.db.captures

        pipeline = [
            {"$match": {"status": "REVIEWING"}},  # Match the specific team
            {"$lookup": {
                "from": "apps",        # The collection to join with
                "localField": "app",  # Field from the input documents
                "foreignField": "_id",    # Field from the "from" documents 
                "as": "app"    # The name of the new array field to add to the input documents
            }},
            {"$unwind": "$app"}
        ]

        populated_captures = list(captures.aggregate(pipeline))
        android_apps = {}
        for capture in populated_captures:
            app = capture["app"]
            if app["os"] == "android":
                package_name = app["metadata"]["name"]
                app_id = app["_id"]
                if package_name not in android_apps:
                    android_apps[package_name] = {
                        "app_id": app_id,
                        "captures": [capture["_id"]]
                    }
                else:
                    android_apps[package_name]["captures"].append(capture["_id"])

        print(len(android_apps))
        return android_apps

    def check_if_app_exists_as_ios(self, app_names: list[str]):
        db = self.client[self.DB_NAME]

        regex_patterns = [{"$regex": name, "$options": "i"} for name in app_names]
        apps = db.apps.find({
            "$or": [{"metadata.name": {"$regex": name, "$options": "i"}} for name in app_names],
            "os": "ios"
        }).to_list()
        print("apps to replace", apps)
        return {app["metadata"]["name"]: app["_id"] for app in apps}
    
    def replace_capture_android_app_with_ios(self, android_ios_id_pairs: list[tuple[ObjectId, ObjectId]]):
        """
        Find the captures with the Android app id (key) and replace it with the id in the ios app (value)
        """
        client = MongoClient(self.ATLAS_URI, **self.ssl_kwargs)
        db = client[self.DB_NAME]

        for android_app_id, ios_app_id in android_ios_id_pairs:
            db.captures.update_many(
                {"app": android_app_id},
                {"$set": {"app": ios_app_id}}
            )


    def check_collection(self):
        client = MongoClient(self.ATLAS_URI, **self.ssl_kwargs)
        db = client[self.DB_NAME]
        
        if 'captures' in db.list_collection_names():
            print("✅ captures exists with", db.captures.count_documents({}), "docs")
            print("🔍 Sample doc:", db.captures.find_one())
        else:
            print("❌ captures collection missing")

    def check_capture_in_status_review(self): 
        client = MongoClient(self.ATLAS_URI, **self.ssl_kwargs)
        db = client[self.DB_NAME]

        caps = db.captures.find({"status": "REVIEWING"}).to_list()
        apps = db.captures.aggregate([{
                "$match": {"status": "REVIEWING"}
            }, {
                "$group": {
                    "_id": "$app",
                    "count": { "$sum": 1}
                }
            }
        ]).to_list()
        
        print(caps[0])

        print("Num captures in REVIEWING:", len(caps))
        print("Num apps in REVIEWING:", len(apps))
        
        review_status = db.status_review
        count = review_status.count_documents({"capture_exists": True})
        print(f"Documents with capture_exists=True: {count}")

    def close_connection(self):
        if self.client:
            self.client.close()
            print("Connection closed")

### Connect Atlas Migrator

In [ ]:
migrator = AtlasMigrator()
if not migrator.connect():
    print("Connection failed")
# migrator.run()


### Check Number of Captures in Review

In [ ]:
# migrator.check_capture_in_status_review()

### Check out number of review captures with Android apps

In [ ]:
android_apps = migrator.check_review_capture_with_android_app()

In [ ]:
for name, details in android_apps.items():
    print(name, f"_id: ObjectId('{details['app_id']}')")
    

In [ ]:
apps_to_replace = [
    "Etsy",
    "Expedia",
    "Zillow",
    "Bleacher Report",
    "Yelp",
    "Fishbrain",
    "Quizlet",
    "SeatGeek",
    "TikTok",
    "adidas",
    "AliExpress",
    "Spotify",
    "Google Chat",
    "Google Maps",
    "Duolingo"
]

ios_apps = migrator.check_if_app_exists_as_ios(apps_to_replace)

for name, id in ios_apps.items():
    print(name, f"_id: ObjectId('{id}')")



In [ ]:
app_pairs = [
    (ObjectId("670e19a287478f2c8b782f40"), ObjectId("689ed339507e92fe6f3de492")),
    (ObjectId("670b9e1387478f2c8b781836"), ObjectId("690393d00299111a39c5c3c9")),
    (ObjectId("670b992187478f2c8b781776"), ObjectId("689ed859507e92fe6f3defdc")),
    (ObjectId("670daca187478f2c8b78241f"), ObjectId("689ede0d507e92fe6f3dfbe0")),
    (ObjectId("670db25387478f2c8b78248a"), ObjectId("689ed337507e92fe6f3de488")),
    (ObjectId("670d418987478f2c8b781f88"), ObjectId("689ed7a4507e92fe6f3dee5a")),
    (ObjectId("670d933d87478f2c8b7821e4"), ObjectId("689ed198507e92fe6f3de132")),
    (ObjectId("670bb17e87478f2c8b781a41"), ObjectId("689edf6c507e92fe6f3dfefb")),
    (ObjectId("670d42f887478f2c8b781fa6"), ObjectId("68afbc5971aecd19f39ae30a")),
    (ObjectId("670b0b0787478f2c8b780e23"), ObjectId("689edbf1507e92fe6f3df79b")),
    (ObjectId("670b01c287478f2c8b780bec"), ObjectId("689ed67e507e92fe6f3debc8")),
    (ObjectId("670bb4b387478f2c8b781aa6"), ObjectId("689edf48507e92fe6f3dfe9b")),
    (ObjectId("670b074e87478f2c8b780d61"), ObjectId("689ed1ff507e92fe6f3de1be")),
    (ObjectId("670b85ed87478f2c8b78147a"), ObjectId("689edae1507e92fe6f3df56a")),
    (ObjectId("670b9e1387478f2c8b781836"), ObjectId("690393d00299111a39c5c3c9")),
    (ObjectId("670b7d7e87478f2c8b781308"), ObjectId("6921f7ac5bdaffcb70ce9e7a"))
]

migrator.replace_capture_android_app_with_ios(app_pairs)


### Close Connection

In [ ]:
migrator.close_connection()